# Week 10 Puzzles — Agentic Workflows & GitHub

> **NS5116 Computational Neuroscience — Spring 2026**

This notebook is divided into two parts:

- **Part 1 — Guided Practice (Puzzles 1–10):** Each puzzle includes a worked solution.
- **Part 2 — Independent Practice (Puzzles 11–20):** Write your own solution in the empty code cells.

> **Note:** These puzzles focus on testing, CI config, and data utility patterns — the Python skills underlying agentic workflows and GitHub automation.

---

## Part 1 — Guided Practice (with solutions)

### Puzzle 1 — Write a Simple Test Function

Write a utility function `remove_outliers(values, n_std=2.0)` and a corresponding test function `test_remove_outliers_basic()` using `assert`.

The test should verify that a known outlier is removed.

In [ ]:
def remove_outliers(values, n_std=2.0):
    """Remove values more than n_std standard deviations from the mean.

    Args:
        values (list[float]): Input values.
        n_std (float): Number of standard deviations for threshold.

    Returns:
        list[float]: Filtered values.
    """
    if not values:
        return []
    mean = sum(values) / len(values)
    variance = sum((x - mean) ** 2 for x in values) / len(values)
    std = variance ** 0.5
    return [x for x in values if abs(x - mean) <= n_std * std]


def test_remove_outliers_basic():
    data = [10, 11, 12, 10, 11, 100]   # 100 is an outlier
    result = remove_outliers(data, n_std=2.0)
    assert 100 not in result, "Outlier 100 should be removed"
    assert all(v in result for v in [10, 11, 12]), "Normal values should be kept"
    print("✓ test_remove_outliers_basic passed")


test_remove_outliers_basic()

### Puzzle 2 — Test an Edge Case: Empty Input

Write a test `test_remove_outliers_empty()` that verifies `remove_outliers([])` returns an empty list without raising an error.

In [ ]:
def test_remove_outliers_empty():
    result = remove_outliers([])
    assert result == [], f"Expected [], got {result}"
    print("✓ test_remove_outliers_empty passed")


test_remove_outliers_empty()

### Puzzle 3 — Test with `pytest.raises` Pattern

Write a function `compute_accuracy(n_correct, n_total)` that raises `ValueError` when `n_total` is 0.

Then write a test using a `try/except` pattern (simulating `pytest.raises`) to verify the exception is raised.

In [ ]:
def compute_accuracy(n_correct, n_total):
    """Compute accuracy as a proportion.

    Raises:
        ValueError: If n_total is 0.
    """
    if n_total == 0:
        raise ValueError("Cannot compute accuracy: n_total is 0")
    return n_correct / n_total


def test_accuracy_zero_total():
    try:
        compute_accuracy(5, 0)
        assert False, "Should have raised ValueError"
    except ValueError as e:
        assert "n_total is 0" in str(e)
    print("✓ test_accuracy_zero_total passed")


def test_accuracy_normal():
    result = compute_accuracy(8, 10)
    assert result == 0.8, f"Expected 0.8, got {result}"
    print("✓ test_accuracy_normal passed")


test_accuracy_zero_total()
test_accuracy_normal()

### Puzzle 4 — Write a Data Cleaning Function and Test It

Write `clean_rt_data(records)` that:
- Removes records with `None` RT
- Converts RT strings to float
- Removes RTs outside [100, 1500] ms

Write two tests to verify the cleaning logic.

In [ ]:
def clean_rt_data(records):
    """Clean a list of trial dicts: convert RT, filter invalid values.

    Args:
        records (list[dict]): Trial records with 'rt' field.

    Returns:
        list[dict]: Cleaned records.
    """
    cleaned = []
    for r in records:
        if r.get("rt") is None:
            continue
        try:
            rt = float(r["rt"])
        except (ValueError, TypeError):
            continue
        if 100 <= rt <= 1500:
            cleaned.append({**r, "rt": rt})
    return cleaned


def test_clean_removes_none():
    records = [{"trial": 1, "rt": 300}, {"trial": 2, "rt": None}]
    result = clean_rt_data(records)
    assert len(result) == 1, f"Expected 1, got {len(result)}"
    print("✓ test_clean_removes_none passed")


def test_clean_filters_range():
    records = [
        {"trial": 1, "rt": "50"},     # too fast
        {"trial": 2, "rt": "400"},    # valid
        {"trial": 3, "rt": "2000"},   # too slow
    ]
    result = clean_rt_data(records)
    assert len(result) == 1
    assert result[0]["rt"] == 400.0
    print("✓ test_clean_filters_range passed")


test_clean_removes_none()
test_clean_filters_range()

### Puzzle 5 — Mock an External API Call

Write a function `fetch_station_count(url)` that calls a URL and returns the number of records in the JSON response.

Then write a test that **mocks** `requests.get` so no real network call is made. Use `unittest.mock.patch`.

In [ ]:
import requests
from unittest.mock import patch, MagicMock

def fetch_station_count(url):
    """Fetch JSON from url and return the number of records."""
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json()
    return len(data.get("records", []))


def test_fetch_station_count_mocked():
    mock_response = MagicMock()
    mock_response.json.return_value = {
        "records": [
            {"station": "Taipei", "pm25": 22},
            {"station": "Kaohsiung", "pm25": 35},
        ]
    }
    mock_response.raise_for_status.return_value = None

    with patch("requests.get", return_value=mock_response):
        count = fetch_station_count("https://fake.api/data")

    assert count == 2, f"Expected 2, got {count}"
    print("✓ test_fetch_station_count_mocked passed")


test_fetch_station_count_mocked()

### Puzzle 6 — Parse a GitHub Actions YAML Workflow

YAML is the config format for GitHub Actions. Parse a simple YAML string into a Python dict and extract the Python version and test command.

Use the `yaml` module (`pip install pyyaml`).

In [ ]:
import yaml

workflow_yaml = """
name: Run tests
on:
  push:
    branches: ["main"]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Run pytest
        run: pytest tests/ -v
"""

workflow = yaml.safe_load(workflow_yaml)

# Extract key information
steps = workflow["jobs"]["test"]["steps"]
python_version = steps[0]["with"]["python-version"]
test_command   = steps[1]["run"]

print(f"Workflow name:  {workflow['name']}")
print(f"Python version: {python_version}")
print(f"Test command:   {test_command}")

### Puzzle 7 — Generate a pytest Test File

Write a function `generate_test_file(filepath, function_name, module_path)` that creates a test file with:
- An import of the function from the module
- A passing test stub
- A failing test stub (placeholder)

Print the generated file content.

In [ ]:
import os

def generate_test_file(filepath, function_name, module_path):
    """Generate a pytest test file skeleton.

    Args:
        filepath (str): Output test file path.
        function_name (str): Name of the function to test.
        module_path (str): Dotted import path (e.g., 'utils.clean').
    """
    code = f'''"""Tests for {module_path}.{function_name}"""

from {module_path} import {function_name}


def test_{function_name}_basic():
    """Test {function_name} with normal input."""
    # TODO: replace with actual test
    result = {function_name}()
    assert result is not None


def test_{function_name}_empty():
    """Test {function_name} with empty input."""
    # TODO: replace with actual test
    pass
'''
    os.makedirs(os.path.dirname(filepath), exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(code)


generate_test_file("tests/test_clean.py", "remove_outliers", "utils.clean")

with open("tests/test_clean.py") as f:
    print(f.read())

import shutil
shutil.rmtree("tests")

### Puzzle 8 — Format a Commit Message

Write a function `format_commit_message(action, component, description)` that produces a conventional commit message.

Format: `action(component): description`  
Example: `feat(data): add earthquake data fetcher`

In [ ]:
def format_commit_message(action, component, description):
    """Format a conventional commit message.

    Args:
        action (str): Commit type (feat, fix, test, docs, refactor).
        component (str): Module or area affected.
        description (str): Short description (imperative mood).

    Returns:
        str: Formatted commit message.
    """
    valid_actions = {"feat", "fix", "test", "docs", "refactor", "chore"}
    if action not in valid_actions:
        raise ValueError(f"Action must be one of {valid_actions}")
    return f"{action}({component}): {description}"


# Test
messages = [
    format_commit_message("feat", "data", "add earthquake data fetcher"),
    format_commit_message("test", "clean", "add outlier removal tests"),
    format_commit_message("fix", "app", "filter chart by selectbox value"),
]

for msg in messages:
    print(msg)

### Puzzle 9 — Generate a Branch Name from a Task Description

Write a function `make_branch_name(prefix, description)` that converts a human-readable description into a git branch name.

Rules: lowercase, spaces replaced by hyphens, max 50 characters, no special characters.

Example: `make_branch_name("feature", "Add earthquake data tab")` → `"feature/add-earthquake-data-tab"`

In [ ]:
import re

def make_branch_name(prefix, description, max_length=50):
    """Convert a task description to a git branch name.

    Args:
        prefix (str): Branch prefix (e.g., 'feature', 'fix').
        description (str): Human-readable description.
        max_length (int): Maximum total length.

    Returns:
        str: Formatted branch name.
    """
    slug = description.lower().strip()
    slug = re.sub(r"[^a-z0-9\s-]", "", slug)   # remove special chars
    slug = re.sub(r"\s+", "-", slug)             # spaces → hyphens
    full = f"{prefix}/{slug}"
    return full[:max_length]


# Test
tests = [
    ("feature", "Add earthquake data tab"),
    ("fix", "Chart doesn't update on selectbox change!"),
    ("test", "Add tests for data cleaning module"),
]

for prefix, desc in tests:
    print(make_branch_name(prefix, desc))

### Puzzle 10 — Run All Tests and Summarize Results

Write a simple test runner `run_tests(test_functions)` that takes a list of test functions, runs each one, catches `AssertionError` for failures, and prints a summary.

Include both passing and deliberately failing tests.

In [ ]:
def run_tests(test_functions):
    """Run a list of test functions and print a summary."""
    passed = 0
    failed = 0
    errors = []

    for test_fn in test_functions:
        try:
            test_fn()
            passed += 1
        except AssertionError as e:
            failed += 1
            errors.append((test_fn.__name__, str(e)))

    print(f"\n{'='*40}")
    print(f"{passed} passed, {failed} failed, {passed + failed} total")
    for name, msg in errors:
        print(f"  FAILED: {name} — {msg}")


# Define tests
def test_pass():
    assert 1 + 1 == 2

def test_fail():
    assert 1 + 1 == 3, "Math is broken"

def test_outlier():
    result = remove_outliers([10, 11, 12, 100])
    assert 100 not in result

run_tests([test_pass, test_fail, test_outlier])

---

## Part 2 — Independent Practice (write your own solutions)

### Puzzle 11 — Test Floating-Point Comparison

Write a function `approx_equal(a, b, tol=1e-6)` and test it with values that would fail exact `==` comparison due to floating-point errors (e.g., `0.1 + 0.2` vs `0.3`).

In [ ]:
# Your solution here


### Puzzle 12 — Parameterised Test Concept

Write a function `test_classify_rt_parametrized()` that tests `classify_rt(rt)` from Week 02 against multiple (input, expected) pairs using a loop.

Test cases: `[(50, "anticipation"), (200, "fast"), (400, "normal"), (800, "slow")]`

In [ ]:
# Your solution here


### Puzzle 13 — Test a CSV Round-Trip

Write a test that:
1. Creates a list of trial dicts
2. Writes them to a CSV file
3. Reads them back
4. Asserts the data matches the original

Use `csv.DictWriter` and `csv.DictReader`. Clean up the file after the test.

In [ ]:
# Your solution here


### Puzzle 14 — Write a Test for a GroupBy Summary Function

Write a function `group_mean_rt(records, group_key)` that groups trial records by `group_key` and returns a dict mapping each group to its mean RT.

Write at least two tests: one for basic grouping and one for a single-group case.

In [ ]:
# Your solution here


### Puzzle 15 — Mock `datetime.date.today()`

Write a function `generate_filename()` that includes today's date in the output filename.

Write a test that mocks `datetime.date.today()` to return a fixed date, so the test is deterministic.

In [ ]:
# Your solution here


### Puzzle 16 — Generate a GitHub Actions Workflow YAML

Write a function `generate_ci_yaml(python_version, test_command, requirements_file)` that returns a valid GitHub Actions YAML string for running tests on push.

Parse the output with `yaml.safe_load()` to verify it is well-formed.

In [ ]:
# Your solution here


### Puzzle 17 — Test a Date Parser

Write a function `parse_date(date_string)` that handles multiple date formats: `"2024-01-15"`, `"01/15/2024"`, and `"Jan 15, 2024"`.

Write three tests, one for each format, all expecting the same `datetime.date` output.

In [ ]:
# Your solution here


### Puzzle 18 — Test Fixture Pattern

Write a `create_sample_data()` fixture function that returns a pre-built list of trial dicts. Use this fixture in two different test functions to avoid duplicating test data setup.

In [ ]:
# Your solution here


### Puzzle 19 — Test That a Function Does Not Mutate Its Input

Write a test that verifies `remove_outliers()` does not modify the original list. Make a copy before calling the function, and assert the original is unchanged after.

In [ ]:
# Your solution here


### Puzzle 20 — Full Test Suite with Runner *(Bonus)*

Combine everything:
1. Define a utility module with 2–3 functions (e.g., `compute_accuracy`, `remove_outliers`, `classify_rt`)
2. Write 6+ test functions covering normal cases, edge cases, and error cases
3. Run all tests with the `run_tests()` runner from Puzzle 10
4. Print a detailed summary with pass/fail counts

In [ ]:
# Your solution here
